# Time Series Practice 2

## Problem Definition — 5W1H

| 5W1H | Nội dung |
|---|---|
| **What** | Khám phá dữ liệu và chuẩn bị chuỗi thời gian cho các exercise tiếp theo. |
| **Why** | Nhận diện trend, seasonality, khả năng non-stationarity và xác lập input contract. |
| **Who** | Chuỗi Monthly Retail Sales trong `a10.csv`. |
| **When** | Dữ liệu monthly từ 1991-07 đến 2008-06. |
| **Where** | `datasets/raw/a10.csv` của Practice 2. |
| **How** | Load, validate, chuyển date thành index, vẽ plot và split chronological 80/20. |

## Table of Contents

- [Exercise 1 — Khám phá dữ liệu](#exercise-1)
- [Exercise 2 — Kiểm định ADF trên chuỗi gốc](#exercise-2)
- [Exercise 3 — Sai phân và kiểm định lại](#exercise-3)
- [Final validation](#final-validation)

## Scientific rules

Raw data bất biến; không shuffle; `date` phải thành `DatetimeIndex`; test set được seal; E01 không chạy ADF, differencing, model training hoặc forecasting.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
from practice_2.data.time_series_data import load_monthly_retail_sales, chronological_split
from practice_2.visualization.plots import save_series_plot

PROJECT_ROOT = next((p for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent] if (p / 'datasets' / 'raw' / 'a10.csv').is_file()), None)
if PROJECT_ROOT is None: raise FileNotFoundError('Không tìm thấy Practice 2 root')
RAW_PATH = PROJECT_ROOT / 'datasets' / 'raw' / 'a10.csv'
FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures'
RESULTS_DIR = PROJECT_ROOT / 'outputs' / 'results'
FIGURES_DIR.mkdir(parents=True, exist_ok=True); RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# Exercise 1 — Khám phá dữ liệu (EDA)
<a id="exercise-1"></a>

Mục tiêu: load dữ liệu, chuyển `date` thành time index, vẽ đồ thị và trả lời trend, seasonality, initial stationarity.

## 1. Nạp dữ liệu

Loader reusable kiểm tra schema `date,value`, parse datetime và bảo toàn raw values trên working representation.

In [ ]:
raw_frame = pd.read_csv(RAW_PATH)
raw_schema = pd.DataFrame({'column': raw_frame.columns, 'dtype': raw_frame.dtypes.astype(str).values})
print('shape:', raw_frame.shape)
raw_frame.info()
display(raw_frame.head())
display(raw_schema)
series = load_monthly_retail_sales(RAW_PATH)
train_series, test_series = chronological_split(series, train_fraction=0.80)
display(series.head().rename('value').to_frame())


## 2. Kiểm tra data contract và time index

Kiểm tra schema, date range, monthly frequency, chronology, missing values, duplicate timestamps và missing intervals.

In [ ]:
expected_index = pd.date_range(series.index.min(), series.index.max(), freq='MS')
validation = pd.DataFrame({'metric':['rows','start','end','index_type','frequency','missing_values','duplicate_timestamps','missing_intervals','chronological'],'value':[len(series),series.index.min(),series.index.max(),type(series.index).__name__,series.index.freqstr,int(series.isna().sum()),int(series.index.has_duplicates),len(expected_index.difference(series.index)),series.index.is_monotonic_increasing]})
boundary = pd.DataFrame({'split':['train','test'],'start':[train_series.index.min(),test_series.index.min()],'end':[train_series.index.max(),test_series.index.max()],'observations':[len(train_series),len(test_series)],'forecast_horizon':[None,len(test_series)]})
validation.to_csv(RESULTS_DIR / 'dataset_validation.csv', index=False); boundary.to_csv(RESULTS_DIR / 'train_test_boundary.csv', index=False)
display(validation); display(boundary)


## 3. Thống kê mô tả

Thống kê mô tả cung cấp evidence về level và variability trước khi nhìn đồ thị. Đây là descriptive evidence, chưa phải stationarity test.

In [ ]:
descriptive = series.describe().to_frame('value').T
rolling_std_first = float(series.iloc[:24].std())
rolling_std_last = float(series.iloc[-24:].std())
q1, q3 = series.quantile([0.25, 0.75])
iqr = q3 - q1
potential_outliers = series[(series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)]
display(descriptive)
display(pd.DataFrame({'metric':['first_24_std','last_24_std','q1','q3','iqr','potential_outlier_count'],'value':[rolling_std_first,rolling_std_last,q1,q3,iqr,len(potential_outliers)]}))


## 4. Vẽ đồ thị chuỗi thời gian

Figure được lưu tại `outputs/figures/01_original_time_series.png`.

In [ ]:
figure_path = FIGURES_DIR / '01_original_time_series.png'
save_series_plot(series, figure_path, 'Monthly Retail Sales — Original Time Series', 'Value')
display(Image(filename=str(figure_path)))


## 5. Trả lời câu hỏi E01

### 5.1. Chuỗi có xu hướng không?
Có. Giá trị tăng từ khoảng **3.53** ở đầu chuỗi lên **19.43** ở cuối chuỗi; mean 24 quan sát đầu khoảng **3.84**, còn 24 quan sát cuối khoảng **21.72**. Đồ thị và thống kê mô tả cùng hỗ trợ kết luận trend tăng.

### 5.2. Có dấu hiệu mùa vụ không?
Có dấu hiệu mùa vụ theo chu kỳ 12 tháng. Monthly means không đồng đều; ví dụ tháng 1 khoảng **14.17** và tháng 2 khoảng **8.61**, cho thấy pattern lặp trong năm. Đây là seasonal observation từ EDA, chưa phải seasonal model.

### 5.3. Chuỗi có khả năng dừng không?
Khả năng cao là **không dừng** vì mean thay đổi theo thời gian và trend tăng rõ. Đây là nhận định EDA ban đầu; ADF ở E02 mới là kiểm định chính thức.

### Limitation
Đồ thị hỗ trợ nhận diện trend/seasonality nhưng không đủ chứng minh stationarity. E01 không chọn `d`, `p`, `q` và không dùng test set để model identification.

<a id="e01-validation"></a>

# E01 validation checkpoint

E01 đã load và validate dataset, chuyển date thành `DatetimeIndex`, kiểm tra monthly intervals, tạo plot, ghi nhận EDA observations và xác lập chronological train/test boundary. Không chạy ADF, differencing, model hoặc forecast trong exercise này.

# Exercise 2 — Kiểm định ADF trên chuỗi gốc
<a id="exercise-2"></a>

Mục tiêu: kiểm định stationarity của original training series bằng Augmented Dickey-Fuller test. E02 không differencing và không dùng test set.

## 1. Phương pháp và giả thuyết

Augmented Dickey-Fuller test kiểm tra liệu chuỗi có unit root hay không. Dạng khái quát của kiểm định là:

$$\Delta y_t = \alpha + \beta t + \gamma y_{t-1} + \sum_{i=1}^{p} \delta_i \Delta y_{t-i} + \varepsilon_t$$

Trong đó, kiểm định tập trung vào hệ số của mức trễ `γ`.

- **H₀:** chuỗi có unit root, tức chưa có tính dừng.
- **H₁:** chuỗi không có unit root, evidence hỗ trợ tính dừng (stationarity).
- Với `alpha = 0.05`: p-value ≤ alpha dẫn đến reject H₀.
- p-value > alpha dẫn đến fail to reject H₀; điều này không phải proof tuyệt đối rằng H₀ đúng.

## 2. ADF input contract và scientific parameters

ADF chạy trên `train_series` nguyên bản từ E01, gồm 163 observations. `test_series` gồm 41 observations và được giữ sealed. Không transformation nào được thực hiện trước phép kiểm định.

| Parameter | Value |
|---|---|
| Test | Augmented Dickey-Fuller |
| Input | Original training series |
| Training observations | 163 |
| Test observations | 41, excluded |
| Significance level | α = 0.05 |
| Autolag | AIC |
| Differencing | Not performed |
| Model training | Not performed |

In [ ]:
from practice_2.analysis.stationarity import run_adf
ADF_ALPHA = 0.05
adf_original, critical_original = run_adf(train_series, alpha=ADF_ALPHA)
adf_original.to_csv(RESULTS_DIR / 'adf_original_series.csv', index=False)
critical_original.to_csv(RESULTS_DIR / 'adf_original_critical_values.csv', index=False)
adf_statistic = float(adf_original.loc[adf_original['metric'].eq('adf_statistic'), 'value'].iloc[0])
p_value = float(adf_original.loc[adf_original['metric'].eq('p_value'), 'value'].iloc[0])
critical_comparison = critical_original.copy()
critical_comparison['statistic_below_critical'] = adf_statistic < critical_comparison['value']
display(adf_original)
display(critical_original)
display(critical_comparison)


## 3. Decision rule

| Condition | Decision | Interpretation |
|---|---|---|
| p-value ≤ 0.05 | Reject H₀ | Evidence supports stationarity |
| p-value > 0.05 | Fail to reject H₀ | Insufficient evidence of stationarity |

## 4. Bằng chứng định lượng

Kết luận phải được đọc từ `adf_original_series.csv` và critical-value table vừa sinh. Báo cáo ADF statistic, p-value, used lag, số observations, alpha và decision. Đồng thời đối chiếu ADF statistic với từng critical value trong bảng comparison.

## 5. Interpretation và trả lời câu hỏi E02

Actual output cho ADF statistic khoảng **2.230893**, p-value khoảng **0.998907**, used lag **14** và **148** observations được sử dụng trong phép test.

1. **H₀ của ADF là gì?** Chuỗi có unit root và chưa có evidence đủ để xem là stationary.
2. **p-value cho biết điều gì?** Mức độ phù hợp của dữ liệu với H₀; p-value nhỏ cung cấp evidence chống lại H₀, không phải xác suất H₀ đúng.
3. **Chuỗi gốc có dừng không?** Vì p-value **0.998907 > 0.05**, decision là **fail to reject H₀**. Statistic cũng không nhỏ hơn các critical values âm. Do đó, chưa đủ evidence để kết luận chuỗi gốc stationary; kết quả phù hợp với nhận định non-stationarity ban đầu ở E01.

### Downstream implication

E02 không tự chọn `d`. Kết quả này cung cấp evidence để E03 thực hiện first-order differencing trên training series, sau đó phải chạy ADF retest. Test set vẫn tiếp tục được giữ sealed.

### E02 limitation

E02 chỉ kiểm định original training series. Không differencing, không ACF/PACF, không model fitting và không forecast.

<a id="e02-validation"></a>
# E02 validation checkpoint

E01 đã hoàn thành EDA và sealed boundary. E02 đã chạy ADF trên original training series, lưu statistic/p-value/critical values và ghi nhận stationarity decision. E03 chưa được triển khai trong notebook này.

# Exercise 3 — Sai phân và kiểm định lại
<a id="exercise-3"></a>

Mục tiêu: thực hiện first-order differencing trên original training series, vẽ differenced series, chạy ADF lần 2 và xác định liệu `d = 1` có được evidence hỗ trợ hay không.

## 1. Input contract và transformation protocol

Input là `train_series` nguyên bản từ E01; E02 đã cho thấy chuỗi gốc fail to reject H₀. First difference chỉ được áp dụng trên working copy. Test set vẫn sealed và raw dataset không bị sửa. Quan sát đầu tiên mất do phép sai phân và được loại khỏi working series.

In [ ]:
from practice_2.analysis.stationarity import first_difference
differenced = first_difference(train_series)
differenced_figure = FIGURES_DIR / '02_first_differenced_series.png'
save_series_plot(differenced, differenced_figure, 'First-Differenced Monthly Retail Sales', 'First difference')
display(Image(filename=str(differenced_figure)))
print('original training observations:', len(train_series))
print('differenced observations:', len(differenced))


## 2. ADF retest và decision rule

ADF lần 2 chạy trên `differenced` với `alpha = 0.05`.

| Condition | Decision | Interpretation |
|---|---|---|
| p-value ≤ 0.05 | Reject H₀ | Evidence supports stationarity after differencing; `d = 1` có căn cứ |
| p-value > 0.05 | Fail to reject H₀ | First difference chưa đủ evidence; không tự tăng differencing |

In [ ]:
adf_diff, critical_diff = run_adf(differenced, alpha=ADF_ALPHA)
diff_statistic = float(adf_diff.loc[adf_diff['metric'].eq('adf_statistic'), 'value'].iloc[0])
diff_p_value = float(adf_diff.loc[adf_diff['metric'].eq('p_value'), 'value'].iloc[0])
selected_d = 1 if diff_p_value <= ADF_ALPHA else 'REVIEW_REQUIRED'
diff_comparison = critical_diff.copy()
diff_comparison['statistic_below_critical'] = diff_statistic < diff_comparison['value']
differencing_decision = pd.concat([adf_diff, pd.DataFrame({'metric':['selected_d'], 'value':[selected_d]})], ignore_index=True)
adf_diff.to_csv(RESULTS_DIR / 'adf_differenced_series.csv', index=False)
critical_diff.to_csv(RESULTS_DIR / 'adf_differenced_critical_values.csv', index=False)
differencing_decision.to_csv(RESULTS_DIR / 'differencing_decision.csv', index=False)
display(differencing_decision)
display(diff_comparison)


## 3. Interpretation và trả lời câu hỏi E03

Actual ADF retest cho statistic khoảng **-1.706074**, p-value khoảng **0.427986**, used lag **13** và **148** observations. Vì p-value lớn hơn 0.05, first difference **chưa đủ evidence để kết luận stationary**.

Do đó, `d` chưa được xác nhận là 1 trong protocol hiện tại và decision được ghi là **REVIEW_REQUIRED**. Không tự động sai phân thêm, không fit model và không chuyển sang E04 khi chưa có protocol được duyệt.

### Limitations và stop condition

ADF evidence là cơ sở thống kê của decision nhưng không phải bằng chứng duy nhất cho mọi loại non-stationarity. Trong phạm vi E03, kết quả chưa đạt được ghi nhận trung thực như một blocker; test set vẫn không được sử dụng.

# Final validation

E01 đã validate dữ liệu và sealed boundary; E02 đã kiểm định chuỗi gốc; E03 đã tạo first difference, plot và ADF retest. E03 đang có scientific blocker vì first difference chưa đạt stationarity, nên chưa chuyển sang ACF/PACF hoặc model fitting.